<a href="https://colab.research.google.com/github/0220nakarina-ui/welfare-scheduling-engine/blob/main/%E3%82%B7%E3%83%95%E3%83%88%E4%BD%9C%E6%88%90%E4%B8%AD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 【要件定義】

# 職員総数8名(責任者は2名)
# 前提 平日は最低5人出勤/土曜と祝日は最低2人
# 日曜は3週目のみ開所
# 日勤のみだが時差出勤がある場合有
# 0.1.2でシフト表記→0:出勤・1:時差出勤・2:休み
# 0と1の人について、役割(ファシリテーター・サポート・リーダー)を各1人に振り分け
# 土日祝についてはファシリテーター1人とサポート・リーダー兼任者1人とする
# 時差出勤について、9:00又は9：30のどちらかとする


# Pythonで作成するが、今後スプレッドシート又はExcelでの操作を想定
# 職員が触るのはセル内のみ、コードは触らないようにする
# 連勤は5日まで
# 職員1人に負担が偏らないようにする
# 印刷して各職員に渡す・事業所フロア内にも同じものを貼り付け
# 1か月のシフト(2,4,6,9.,11月は31日目が無いのと年末年始は閉所)
# 閉所日についてはスプレッドシート又はExcelで職員に打ち込んでもらう
# 該当の年と月も同様だが、これについては自動更新

In [ ]:
# 【全体設計】

# ①Pythonでの役割
# シフト自動生成
# 制約チェック(人数や連勤)
# ファイル出力

# ②スプレッドシート又はExcelの役割
# 職員が直接編集する
# 閉所日や微調整(希望休や有休・時差出勤等)
# 印刷

In [ ]:
# シフト作成

!pip install pandas openpyxl

import pandas as pd
import random
import calendar
from collections import defaultdict
from google.colab import files
from typing import Dict, List, Tuple
import io

# Excelをアップロード
print("Excelをアップロードしてください")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

xl = pd.ExcelFile(file_name)

staff_df = xl.parse("職員リスト")
config_df = xl.parse("設定")

staff = staff_df["名前"].tolist()
leaders = set(staff_df[staff_df["リーダー"] == 1]["名前"].tolist())

YEAR = int(config_df.loc[0, "値"])
MONTH = int(config_df.loc[1, "値"])

closed_days_raw = str(config_df.loc[2, "値"])
closed_days = set(int(x) for x in closed_days_raw.split(",") if x)

# 日曜日の判定
def get_sundays():
    c = calendar.Calendar()
    return [d[0] for d in c.itermonthdays2(YEAR, MONTH) if d[0] != 0 and d[1] == 6]

sundays = get_sundays()
third_sunday = sundays[2] if len(sundays) >= 3 else None

# 必要人数(土日祝は原則2人)
def required(day):
    wd = calendar.weekday(YEAR, MONTH, day)

    if day in closed_days:
        return 0
    if wd == 6:
        return 2 if day == third_sunday else 0
    if wd == 5:
        return 2
    return 5


# シフト作成
def generate_shift():
  days = calendar.monthrange(YEAR,MONTH)[1]

  schedule = {}
  work_count = defaultdict(int)
  consecutive = defaultdict(int)

  for day in range(1,days+1):
    required = required_staff(YEAR,MONTH,day)

    if required == 0:
      schedule[day] = {s: 2 for s in staff}
      for s in staff:
          consecutive[s] = 0
      continue

    candidates = [s for s in staff if consecutive[s] < 5]
    candidates.sort(key=lambda x: (work_count[x], consecutive[x]))

    workers = candidates[:required]

# リーダー確保
    if leaders and not any(w in leaders for w in workers):
                for c in candidates:
                    if c in leaders:
                        workers[-1] = c
                        break

    schedule[d] = {}

    for s in staff:
       if s in workers:
         schedule[d][s] = random.choice([0,1])
         work_count[s] += 1
         consecutive[s] += 1
    else:
        schedule[d][s] = 2
        consecutive[s] = 0

return schedule, work_coun

# 役割
def roles(schedule):
    out = []

    for d, sft in schedule.items():
        workers = [k for k,v in sft.items() if v in [0,1]]
        wd = calendar.weekday(YEAR, MONTH, d)

        if len(workers) == 0:
            out.append({"日付": d})
            continue

        if wd >= 5:
            if len(workers) >= 2:
                fac = random.choice(workers)
                oth = random.choice([w for w in workers if w != fac])
                out.append({"日付": d, "ファシリ": fac, "兼任": oth})
            else:
                out.append({"日付": d})
        else:
            if len(workers) >= 3:
                fac = random.choice(workers)
                sup = random.choice([w for w in workers if w != fac])
                lead = random.choice(list(leaders & set(workers))) if leaders & set(workers) else fac

                out.append({
                    "日付": d,
                    "ファシリ": fac,
                    "サポート": sup,
                    "リーダー": lead
                })
            else:
                out.append({"日付": d})

    return pd.DataFrame(out)

# チェック
def check(schedule):
    rows = []

    for d, sft in schedule.items():
        workers = [v for v in sft.values() if v in [0,1]]
        req = required(d)

        rows.append({
            "日付": d,
            "必要": req,
            "実数": len(workers),
            "判定": "OK" if req == len(workers) else "NG"
        })

    return pd.DataFrame(rows)

# 出力
def export(schedule, roles_df, check_df, wc):
    rows = []

    for d, sft in schedule.items():
        rows.append({
            "日付": d,
            "曜日": ["月","火","水","木","金","土","日"][calendar.weekday(YEAR, MONTH, d)],
            **sft
        })

    df = pd.DataFrame(rows)
    sumdf = pd.DataFrame(list(wc.items()), columns=["職員","出勤数"])

    with pd.ExcelWriter("shift_fast.xlsx") as w:
        df.to_excel(w, sheet_name="シフト", index=False)
        roles_df.to_excel(w, sheet_name="役割", index=False)
        check_df.to_excel(w, sheet_name="チェック", index=False)
        sumdf.to_excel(w, sheet_name="集計", index=False)

# 実行
schedule, wc = generate()
roles_df = roles(schedule)
check_df = check(schedule)

export(schedule, roles_df, check_df, wc)

print("完成しました（チェック・集計付き）")

files.download("shift_fast.xlsx")

Excelをアップロードしてください


KeyboardInterrupt: 